# Evaluation Notebook - Stage 2 vs Stage 3

**Datasets cần attach:**
1. Code project v6+ (có restore_pipeline.py + evaluate_pipeline.py)
2. vn-pretrained-model-gan
3. Stage 2 checkpoint
4. Stage 3 checkpoint (vn-stage3-final-completed)
5. CelebA-HQ test set


## 1. Verify datasets


In [ ]:
from pathlib import Path
print('=== Input datasets ===')
for p in sorted(Path('/kaggle/input').iterdir()):
    n = sum(1 for _ in p.rglob('*') if _.is_file())
    print(f'  {p.name:<45} ({n} files)')

## 2. Setup code project


In [ ]:
import os, sys, shutil
from pathlib import Path

DST = '/kaggle/working/vn-old-portrait-restoration'
SRC = None
for p in Path('/kaggle/input').iterdir():
    hits = list(p.rglob('src/pipeline/restore_pipeline.py'))
    if hits:
        SRC = str(hits[0].parent.parent.parent)
        break
assert SRC, 'Code project (chứa restore_pipeline.py) không tìm thấy. Upload code v6+.'

if os.path.exists(DST):
    shutil.rmtree(DST)
shutil.copytree(SRC, DST)
os.chdir(DST)
sys.path.insert(0, DST)

required = [
    'src/pipeline/restore_pipeline.py',
    'scripts/evaluate_pipeline.py',
    'src/degradation/vietnam_pipeline.py',
]
for f in required:
    assert Path(f).exists(), f'MISSING: {f}'
    print(f'  ✓ {f}')
print(f'\n✓ Code ready at {DST}')

## 3. Install deps + patch GFPGAN

Lưu ý: phải patch basicsr import nếu chưa làm.


In [ ]:
!pip install -q einops basicsr facexlib gfpgan lpips scikit-image
!pip install -q gfpgan==1.3.8 basicsr==1.4.2 facexlib==0.3.0 --force-reinstall --no-deps

import glob
candidates = glob.glob('/usr/local/lib/python*/dist-packages/basicsr/data/degradations.py')
assert candidates, 'basicsr not found'
degradations_file = candidates[0]

with open(degradations_file) as f:
    content = f.read()

old = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
new = 'from torchvision.transforms.functional import rgb_to_grayscale'

if old in content:
    content = content.replace(old, new)
    with open(degradations_file, 'w') as f:
        f.write(content)
    print(f'✓ Patched {degradations_file}')
else:
    print('  (already patched)')

import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
try:
    from gfpgan import GFPGANer
    print('✓ GFPGAN import OK')
except Exception as e:
    print(f'✗ GFPGAN fail: {e}')

## 4. RESTART KERNEL → run lại cells 1-3 → tiếp tục từ đây


## 5. Find checkpoint paths


In [ ]:
from pathlib import Path
import torch

stage2_list = [p for p in Path('/kaggle/input').glob('**/best.pth')
               if p.stat().st_size >= 100_000_000]

STAGE2_CKPT = None
STAGE3_CKPT = None

for p in stage2_list:
    try:
        ckpt = torch.load(p, map_location='cpu', weights_only=False)
        stage = ckpt.get('stage', None)
        fusion_keys = [k for k in ckpt.get('fusion', {}).keys() if 'blocks.' in k]
        block_indices = set()
        for k in fusion_keys:
            try:
                idx = int(k.split('blocks.')[1].split('.')[0])
                block_indices.add(idx)
            except: pass
        num_blocks = max(block_indices) + 1 if block_indices else 0
        
        if stage == 3 or num_blocks >= 3:
            STAGE3_CKPT = str(p)
            print(f'  Stage 3 ckpt: {p}  ({num_blocks} fusion blocks)')
        else:
            STAGE2_CKPT = str(p)
            print(f'  Stage 2 ckpt: {p}  ({num_blocks} fusion blocks)')
    except Exception as e:
        print(f'  ⚠ Skip {p}: {e}')

assert STAGE2_CKPT, 'Stage 2 ckpt not found'
assert STAGE3_CKPT, 'Stage 3 ckpt not found'

GFPGAN_PRE = next(Path('/kaggle/input').rglob('GFPGANv1.4.pth'), None)
assert GFPGAN_PRE, 'GFPGAN pretrained not found'
GFPGAN_PRE = str(GFPGAN_PRE)

CELEBA = None
for p in Path('/kaggle/input').rglob('*'):
    if p.is_dir() and 'celeb' in p.name.lower():
        n_img = sum(1 for _ in p.rglob('*.jpg')) + sum(1 for _ in p.rglob('*.png'))
        if n_img > 1000:
            CELEBA = str(p)
            break
assert CELEBA, 'CelebA-HQ not found'

print(f'\nStage 2:   {STAGE2_CKPT}')
print(f'Stage 3:   {STAGE3_CKPT}')
print(f'GFPGAN:    {GFPGAN_PRE}')
print(f'CelebA:    {CELEBA}')

## 6. Run evaluation

~10-15 phút trên 200 ảnh × 2 levels × 2 stages.


In [ ]:
from scripts.evaluate_pipeline import run_evaluation

results = run_evaluation(
    stage2_ckpt=STAGE2_CKPT,
    stage3_ckpt=STAGE3_CKPT,
    gfpgan_pretrained=GFPGAN_PRE,
    test_dir=CELEBA,
    output_dir='/kaggle/working/eval_results',
    n_samples=200,
    levels=['medium', 'hard'],
    image_size=256,
    save_samples=10,
    device='cuda',
)

## 7. Verify output files


In [ ]:
from pathlib import Path
print('=== Output ===')
for f in sorted(Path('/kaggle/working/eval_results').rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to("/kaggle/working/eval_results")}  ({f.stat().st_size/1024:.1f} KB)')

## 8. View qualitative samples


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

samples_dir = Path('/kaggle/working/eval_results/samples')
samples = sorted(samples_dir.glob('*.png'))[:6]

fig, axes = plt.subplots(len(samples), 1, figsize=(16, 3*len(samples)))
if len(samples) == 1:
    axes = [axes]
for ax, p in zip(axes, samples):
    ax.imshow(Image.open(p))
    ax.set_title(p.stem + '  |  LQ  →  Stage 2  →  Stage 3  →  HQ', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/eval_results/qualitative_grid.png', dpi=80, bbox_inches='tight')
plt.show()

## 9. Backup lên Kaggle Dataset (optional)


In [ ]:
import os, json

os.environ['KAGGLE_USERNAME'] = 'mrtaintou'
os.environ['KAGGLE_KEY'] = 'PASTE_KEY_HERE'

with open('/kaggle/working/eval_results/dataset-metadata.json', 'w') as f:
    json.dump({
        'title': 'VN Stage 2 vs 3 Evaluation Results',
        'id': 'mrtaintou/vn-eval-stage2-stage3',
        'licenses': [{'name': 'CC0-1.0'}]
    }, f)

!kaggle datasets create -p /kaggle/working/eval_results --dir-mode zip